# Credit Card Fraud Detection 2.0

This clean notebook explains and reproduces the V2 model workflow. The original baseline notebook's executable source code is unchanged, but its outputs and metadata have local modifications. Its original committed version remains available in Git history. The Phase 1 audit notebook remains separate. No data is removed. The implementation lives in `fraud_pipeline_v2.py`; importing it here avoids having two copies of the training logic to maintain.

## What this workflow does

1. Reads the transaction CSV and groups rows with identical predictor values.
2. Creates fixed-seed, stratified train/validation/test splits at the group level.
3. Fits leakage-safe preprocessing and three class-imbalance-aware models.
4. Compares models on validation metrics and chooses by PR-AUC (average precision), not accuracy.
5. Chooses the winning model's threshold using validation F1.
6. Evaluates that selected model and threshold on the untouched test partition once.

The notebook is designed for VS Code with the project's `.venv` selected as the kernel.

## 1. Run the workflow in memory

The code cell below trains the models and reproduces the evaluation. `save_artifacts=False` prevents it from replacing the saved files; use `fraud_pipeline_v2.py` in the integrated terminal to create the saved model, CSV, metadata, and charts.

In [ ]:
from IPython.display import display, Image
import pandas as pd
from fraud_pipeline_v2 import run_workflow

results = run_workflow(save_artifacts=False, make_figures=False)
display(results["comparison"].round(5))

## 2. Why groups and stratification are both used

A random row split can put an exact repeated transaction feature row in both training and test data. Grouping by all predictors (excluding the target) keeps each identical feature pattern together. We stratify the unique groups by their label to keep fraud represented in all partitions. Group sizes vary, so the final row percentages may differ slightly; the code reports actual fraud counts and checks there is no group overlap.

In [ ]:
display(pd.DataFrame(results["split_summary"]).T)

## 3. Leakage-safe preprocessing

`Amount` uses `log1p` to reduce its long right tail, followed by a standard scaler. `Time` is standardized. Both fitted transforms are inside each model's scikit-learn pipeline, so their parameters are learned from training data only. `V1`–`V28` already contain transformed numeric values, so they pass through unchanged.

## 4. Class imbalance and candidate models

Only 492 of 284,807 transactions are fraud. Logistic Regression and Random Forest use class weights; XGBoost uses a positive-class weight computed from the training partition. These options make fraud errors matter more during training without altering validation or test rows. The comparison is on validation data, with Precision, Recall, F1, ROC-AUC, PR-AUC, false positives, false negatives, and confusion counts.

## 5. Threshold choice and test-set discipline

A probability threshold converts a fraud score into an alert. The default is 0.5. A lower threshold can catch more fraud while increasing false alarms. This workflow selects a threshold that maximizes F1 on validation data (ties favor the higher threshold). It then evaluates the selected model and threshold on test data once; the test set does not select the model or threshold.

In [ ]:
print("Selected model:", results["selected_model"])
print("Validation-selected threshold:", results["selected_threshold"])
print("Final selected-model test metrics:")
display(pd.Series(results["test_metrics"]).to_frame("value"))

## 6. Saved run results and visualizations

The saved workflow run selected Random Forest by validation PR-AUC (0.8412). At the selected threshold 0.339756, final test results were Precision 0.9474, Recall 0.7500, F1 0.8372, ROC-AUC 0.9388, and PR-AUC 0.8182. It produced 4 false positives and 24 false negatives. These results show a useful alert tradeoff, but the missed fraud count still matters.

In [ ]:
display(Image(filename="figures/model_comparison_dashboard_v2.png"))
display(Image(filename="figures/validation_threshold_tradeoff_v2.png"))
display(Image(filename="figures/validation_confusion_matrices_v2.png"))
display(Image(filename="figures/test_confusion_matrix_v2.png"))

## 7. Saved model and metadata

`models/fraud_detection_pipeline_v2.joblib` contains both preprocessing and the selected Random Forest. `models/fraud_detection_metadata_v2.json` records the chosen threshold, data split, model settings, dataset checksum, and final test metrics. The script reloads the saved model and checks predictions on sample transactions before finishing. See `README_V2.md` and `WINDOWS_EXECUTION_GUIDE_V2.md` for the complete project notes and commands.